# Collect stock news - Cafef

In [ ]:
# Chạy trên Google Colab
import requests
from bs4 import BeautifulSoup
import csv
import time
import re
from datetime import datetime
from urllib.parse import urljoin, quote
from google.colab import drive
import os
import json

# ------------------ CẤU HÌNH ------------------
KEYWORDS = ["VNM", "Vinamilk"]   # danh sách keyword có thể thay đổi
FROM_DATE_STR = "01/01/2023"
TO_DATE_STR   = "30/06/2024"
FROM_DATE = datetime.strptime(FROM_DATE_STR, "%d/%m/%Y")
TO_DATE   = datetime.strptime(TO_DATE_STR, "%d/%m/%Y")

# Drive path để lưu file + folder checkpoint
drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/cafef_scrape"
os.makedirs(OUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUT_DIR, f"cafef_{'_'.join([k.replace(' ','') for k in KEYWORDS])}_{FROM_DATE_STR.replace('/','')}_{TO_DATE_STR.replace('/','')}.csv")

# Cấu hình crawler
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}
TITLE_SELECTOR = "#admWrapsite > div.main > div.detail-section.adm-mainsection > div.wap-content.wp1100 > div > div.left_cate.totalcontentdetail > h1"
DATE_SELECTOR  = "#admWrapsite > div.main > div.detail-section.adm-mainsection > div.wap-content.wp1100 > div > div.left_cate.totalcontentdetail > div.sharemxh.topshare > p > span"

PAGES_PER_CHECKPOINT = 5
MAX_PAGES = None  # None để crawl hết

# ------------------ Helper ------------------
def save_checkpoint(meta, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

def load_checkpoint(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

def append_rows_to_csv(path, rows):
    header = ["Keyword", "Tiêu đề", "Ngày đăng", "URL"]
    file_exists = os.path.exists(path)
    with open(path, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(header)
        writer.writerows(rows)

def scrape_article_get_date_and_title(url):
    try:
        res = requests.get(url, headers=HEADERS, timeout=20)
    except:
        return None
    if res.status_code != 200:
        return None
    soup = BeautifulSoup(res.content, "html.parser")
    title_el = soup.select_one(TITLE_SELECTOR)
    date_el = soup.select_one(DATE_SELECTOR)
    if not title_el or not date_el:
        return None
    title = title_el.get_text(strip=True)
    date_text = date_el.get_text(" ", strip=True)
    m = re.search(r"\d{2}[-/]\d{2}[-/]\d{4}", date_text)
    if not m:
        return None
    date_raw = m.group()
    fmt = "%d-%m-%Y" if "-" in date_raw else "%d/%m/%Y"
    try:
        date_obj = datetime.strptime(date_raw, fmt)
    except:
        return None
    return title, date_obj

# ------------------ Main crawling ------------------
for keyword in KEYWORDS:
    safe_name = keyword.replace(" ", "")
    checkpoint_path = os.path.join(OUT_DIR, f"cafef_{safe_name}_checkpoint.json")

    print(f"\n--- Bắt đầu crawl keyword: {keyword} ---")

    meta = load_checkpoint(checkpoint_path)
    page = meta.get("last_page", 1) if meta else 1
    seen_urls = set(meta.get("seen_urls", [])) if meta else set()
    pages_since_checkpoint = 0
    total_saved = 0

    while True:
        if MAX_PAGES and page > MAX_PAGES:
            break
        search_url = f"https://cafef.vn/tim-kiem/trang-{page}.chn?keywords={quote(keyword)}"
        print(f"\n[SEARCH] Trang {page}: {search_url}")
        try:
            r = requests.get(search_url, headers=HEADERS, timeout=20)
        except:
            break
        if r.status_code != 200:
            break
        soup = BeautifulSoup(r.content, "html.parser")
        h3s = soup.find_all("h3")
        if not h3s:
            break

        urls_in_page = []
        for h3 in h3s:
            a = h3.find("a")
            if a and a.get("href"):
                full = urljoin("https://cafef.vn", a.get("href"))
                if full not in seen_urls:
                    urls_in_page.append(full)
                    seen_urls.add(full)

        if not urls_in_page:
            break

        page_rows = []
        count_out_of_range = 0
        for u in urls_in_page:
            res = scrape_article_get_date_and_title(u)
            if not res:
                time.sleep(0.4)
                continue
            title, date_obj = res
            if date_obj < FROM_DATE:
                count_out_of_range += 1
            elif date_obj > TO_DATE:
                continue
            else:
                page_rows.append([keyword, title, date_obj.strftime("%d/%m/%Y"), u])
            time.sleep(0.5)

        if page_rows:
            append_rows_to_csv(OUTPUT_FILE, page_rows)
            total_saved += len(page_rows)
            print(f"  -> Lưu {len(page_rows)} bài từ trang {page} (tổng {total_saved})")

        pages_since_checkpoint += 1

        # Dừng nếu tất cả bài trên page cũ hơn FROM_DATE
        if count_out_of_range == len(urls_in_page):
            print("  -> Tất cả bài trong trang đều cũ hơn FROM_DATE, dừng paginate.")
            save_checkpoint({"last_page": page, "seen_urls": list(seen_urls)}, checkpoint_path)
            break

        # Lưu checkpoint mỗi PAGES_PER_CHECKPOINT trang
        if pages_since_checkpoint >= PAGES_PER_CHECKPOINT:
            save_checkpoint({"last_page": page, "seen_urls": list(seen_urls)}, checkpoint_path)
            pages_since_checkpoint = 0

        page += 1
        time.sleep(1.0)

    print(f"--- Kết thúc crawl keyword: {keyword}. Tổng bài lưu: {total_saved} ---")

# Collect stock news - tinnhanhck

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import re
from datetime import datetime
from urllib.parse import urljoin
from google.colab import drive
import os

# ================= CẤU HÌNH =================
STOCK_SYMBOL = "VNM"  # Mã chứng khoán
FROM_DATE_STR = "01/01/2023"
TO_DATE_STR   = "30/06/2024"

# Mount Google Drive
drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/stock_crawl_data"
os.makedirs(OUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUT_DIR, f"tinnhanhck_{STOCK_SYMBOL}_{FROM_DATE_STR.replace('/','')}_{TO_DATE_STR.replace('/','')}.csv")

# Chuyển chuỗi thành datetime
FROM_DATE = datetime.strptime(FROM_DATE_STR, "%d/%m/%Y")
TO_DATE   = datetime.strptime(TO_DATE_STR, "%d/%m/%Y")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.google.com/"
}

# ================= HÀM HỖ TRỢ =================
def parse_date(date_text):
    if not date_text: return None
    date_text = date_text.strip().lower()
    patterns = [r"\d{2}/\d{2}/\d{4}", r"\d{2}-\d{2}-\d{4}", r"\d{4}-\d{2}-\d{2}"]
    for pat in patterns:
        m = re.search(pat, date_text)
        if m:
            d_str = m.group()
            for fmt in ["%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d"]:
                try:
                    return datetime.strptime(d_str, fmt)
                except:
                    continue
    # Nếu dạng "2 giờ trước", "vừa xong"
    if any(x in date_text for x in ['trước', 'vừa', 'phút', 'giờ', 'giây']):
        return datetime.now()
    return None

def append_to_csv(rows):
    file_exists = os.path.exists(OUTPUT_FILE)
    with open(OUTPUT_FILE, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Source", "Mã CK", "Tiêu đề", "Ngày đăng", "URL"])
        writer.writerows(rows)

def get_soup(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=20, verify=False)
        if r.status_code == 200:
            return BeautifulSoup(r.content, "html.parser")
    except Exception as e:
        print(f"[ERR] Lỗi kết nối {url}: {e}")
    return None

# ================= CRAWL TinNhanhCK =================
def crawl_tinnhanhck():
    source = "Tinnhanhchungkhoan"
    print(f"\n--- Bắt đầu crawl {source} ---")
    page = 1
    while page <= 20:  # crawl tối đa 20 trang
        url = f"https://tinnhanhchungkhoan.vn/tim-kiem.html?q={STOCK_SYMBOL}&page={page}"
        print(f"  [Page {page}] {url}")
        soup = get_soup(url)
        if not soup: break

        articles = soup.select("article.story")
        if not articles:
            print("    -> Hết kết quả.")
            break

        rows = []
        old_count = 0

        for art in articles:
            a_tag = art.select_one("h2 a") or art.select_one("h3 a")
            if not a_tag: continue
            link = urljoin("https://tinnhanhchungkhoan.vn", a_tag.get('href'))
            title = a_tag.get_text(strip=True)

            time_el = art.select_one("time")
            date_obj = parse_date(time_el.get('datetime') or time_el.get_text(strip=True)) if time_el else None

            # Nếu không lấy được ngày, vào chi tiết
            if not date_obj:
                d_soup = get_soup(link)
                if d_soup:
                    t_el = d_soup.select_one(".article__meta time")
                    if t_el: date_obj = parse_date(t_el.get_text(strip=True))

            if date_obj:
                if FROM_DATE <= date_obj <= TO_DATE:
                    print(f"    ✅ {title} ({date_obj.strftime('%d/%m/%Y')})")
                    rows.append([source, STOCK_SYMBOL, title, date_obj.strftime("%d/%m/%Y"), link])
                elif date_obj < FROM_DATE:
                    old_count += 1

        if rows: append_to_csv(rows)
        if old_count == len(articles):
            print("    ⏹ Tất cả bài trong trang đều quá cũ. Dừng.")
            break

        page += 1
        time.sleep(1)

# ================= MAIN =================
if _name_ == "_main_":
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    print(f"Mã: {STOCK_SYMBOL} | Từ: {FROM_DATE_STR} đến {TO_DATE_STR}")
    print(f"File lưu: {OUTPUT_FILE}")

    try:
        crawl_tinnhanhck()
    except Exception as e:
        print(f"Tinnhanhck Error: {e}")

    print("\n--- HOÀN THÀNH ---")

# Merge all stock news data

In [ ]:
# ======================================================
# Merge tất cả file news CSV từ folder
# ======================================================
from google.colab import drive
import pandas as pd
import os
from glob import glob

# --- Mount Google Drive ---
drive.mount('/content/drive')

# --- Cấu hình ---
FOLDER = "/content/drive/MyDrive/DATA FINAL"  # Folder chứa các CSV
OUTPUT = "/content/drive/MyDrive/AAData_news_final_merged.csv"  # File output

# --- Tìm tất cả file CSV ---
all_files = glob(os.path.join(FOLDER, "**/*.csv"), recursive=True)
print(f"🔹 Tìm thấy {len(all_files)} file CSV.")

if len(all_files) == 0:
    raise ValueError("❌ Không tìm thấy file CSV nào. Kiểm tra lại đường dẫn FOLDER.")

# --- Đọc và merge tất cả file ---
all_dfs = []

for f in all_files:
    try:
        # Đọc file CSV
        df = pd.read_csv(f, encoding="utf-8-sig")

        # Kiểm tra các cột cần thiết
        required_cols = ["Tiêu đề", "Ngày đăng", "URL"]
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️ File {os.path.basename(f)} thiếu cột bắt buộc, bỏ qua.")
            continue

        # Xử lý cột Keyword
        if "Keyword" in df.columns:
            # Nếu có cột Keyword, giữ nguyên
            keyword_col = df["Keyword"].copy()
        elif "Company" in df.columns:
            # Nếu không có Keyword nhưng có Company, dùng Company
            keyword_col = df["Company"].copy()
        else:
            # Nếu không có cả hai, thử lấy từ tên file
            basename = os.path.basename(f)
            parts = basename.split("_")
            if len(parts) >= 2:
                company = parts[1]
            else:
                company = ""
            keyword_col = pd.Series([company] * len(df))

        # Tạo DataFrame mới với 4 cột cần thiết
        new_df = pd.DataFrame({
            "Keyword": keyword_col,
            "Tiêu đề": df["Tiêu đề"],
            "Ngày đăng": df["Ngày đăng"],
            "URL": df["URL"]
        })

        all_dfs.append(new_df)
        print(f"✓ Đọc thành công: {os.path.basename(f)} ({len(new_df)} dòng)")

    except Exception as e:
        print(f"⚠️ Lỗi đọc file {os.path.basename(f)}: {e}")

if len(all_dfs) == 0:
    raise ValueError("❌ Không có file nào được đọc thành công.")

# --- Merge tất cả DataFrame ---
merged_df = pd.concat(all_dfs, ignore_index=True)

# --- Xử lý cột "Ngày đăng" ---
# Thử convert sang datetime để sort (nếu cần)
merged_df["Ngày đăng_temp"] = pd.to_datetime(
    merged_df["Ngày đăng"],
    format="%d/%m/%Y",
    errors="coerce"
)

# --- Loại bỏ dòng trùng lặp ---
before_drop = len(merged_df)
merged_df = merged_df.drop_duplicates(subset=["Tiêu đề", "URL"], keep="first")
after_drop = len(merged_df)
print(f"\n🔹 Đã loại bỏ {before_drop - after_drop} dòng trùng lặp.")

# --- Sắp xếp theo Keyword và Ngày ---
valid_dates = merged_df["Ngày đăng_temp"].notna()
if valid_dates.sum() > 0:
    merged_df = merged_df.sort_values(by=["Keyword", "Ngày đăng_temp"], ascending=[True, False])
else:
    merged_df = merged_df.sort_values(by="Keyword")

# Xóa cột tạm
merged_df = merged_df.drop(columns=["Ngày đăng_temp"])

# --- Reset index ---
merged_df = merged_df.reset_index(drop=True)

# --- Lưu file CSV ---
merged_df.to_csv(OUTPUT, index=False, encoding="utf-8-sig")

print(f"\n✅ Hoàn tất! File merged đã được lưu tại:")
print(f"   {OUTPUT}")
print(f"\n📊 Thống kê:")
print(f"   - Tổng số file đọc: {len(all_dfs)}")
print(f"   - Tổng số dòng: {len(merged_df)}")
print(f"   - Các cột: {list(merged_df.columns)}")

# Thống kê theo Keyword
print(f"\n📈 Số lượng bài viết theo Keyword:")
keyword_counts = merged_df["Keyword"].value_counts()
print(keyword_counts)

print("\n--- 5 dòng đầu tiên ---")
print(merged_df.head())

print("\n--- 5 dòng cuối cùng ---")
print(merged_df.tail())

# Collect stock price, log return, market return, volume

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime
from google.colab import drive

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# ============================================================
# CẤU HÌNH
# ============================================================
vn30_tickers = [
    "ACB","BCM","BID","BVH","CTG","FPT","GAS","GVR","HDB","HPG","LPB",
    "MBB","MSN","MWG","PLX","SAB","SHB","SSB","SSI","STB","TCB","TPB",
    "VCB","VHM","VIB","VIC","VJC","VNM","VPB","VRE"
]

start_date_str = "2023-01-01"
end_date_str = "2024-06-30"

# ============================================================
# HÀM HỖ TRỢ GỌI API TRỰC TIẾP (KHÔNG CẦN THƯ VIỆN)
# ============================================================
def convert_date_to_timestamp(date_str):
    """Chuyển ngày YYYY-MM-DD sang Epoch timestamp"""
    return int(time.mktime(datetime.strptime(date_str, "%Y-%m-%d").timetuple()))

def get_data_tcbs(ticker, start, end, type='stock'):
    """Gọi trực tiếp API của TCBS để lấy dữ liệu lịch sử"""
    start_ts = convert_date_to_timestamp(start)
    end_ts = convert_date_to_timestamp(end)

    url = f"https://apipubaws.tcbs.com.vn/stock-insight/v1/stock/bars-long-term?ticker={ticker}&type={type}&resolution=D&from={start_ts}&to={end_ts}"

    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        df = pd.DataFrame(data['data'])

        if df.empty:
            return None

        # Đổi tên cột cho chuẩn
        df = df.rename(columns={
            'tradingDate': 'Date',
            'close': 'Closing price',
            'volume': 'Volume'
        })

        # Xử lý ngày tháng (API trả về 2023-01-01T00:00:00.000Z)
        df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

        # Chỉ lấy cột cần thiết
        return df[['Date', 'Closing price', 'Volume']]

    except Exception as e:
        print(f"Lỗi khi tải {ticker}: {e}")
        return None

# ============================================================
# 2. TẢI VN-INDEX (MARKET RETURN)
# ============================================================
print("⏳ Đang tải VN-INDEX...")
df_market = get_data_tcbs("VNINDEX", start_date_str, end_date_str, type='index')

if df_market is not None:
    df_market['Date'] = pd.to_datetime(df_market['Date'])
    df_market = df_market.sort_values('Date')

    # Tính Market Return
    df_market['Market return'] = np.log(df_market['Closing price'] / df_market['Closing price'].shift(1))

    # Chuẩn bị để merge
    df_market = df_market[['Date', 'Market return']].dropna()
    # Chuyển về string để merge chính xác
    df_market['Date'] = df_market['Date'].dt.strftime('%Y-%m-%d')
    print("✔ Tải VN-INDEX thành công!")
else:
    print("❌ Không tải được VN-INDEX")
    df_market = pd.DataFrame(columns=['Date', 'Market return'])

# ============================================================
# 3. TẢI 30 CỔ PHIẾU
# ============================================================
print(f"⏳ Đang tải dữ liệu {len(vn30_tickers)} mã cổ phiếu...")
all_data = []

for symbol in vn30_tickers:
    df = get_data_tcbs(symbol, start_date_str, end_date_str)

    if df is not None:
        # Tính toán
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date')

        # Tính Log Return
        df['Log return'] = np.log(df['Closing price'] / df['Closing price'].shift(1))

        # Thêm tên công ty
        df['Company'] = symbol

        # Chuyển Date về string để merge
        df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

        all_data.append(df)
        print(f"✔ {symbol}: {len(df)} dòng")
    else:
        print(f"⚠ Không có dữ liệu: {symbol}")

# ============================================================
# 4. GỘP VÀ LƯU FILE
# ============================================================
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)

    # Merge với Market Return
    final_df = final_df.merge(df_market, on='Date', how='left')

    # Xóa dòng thiếu dữ liệu (do tính log ngày đầu tiên)
    final_df = final_df.dropna()

    # Sắp xếp
    final_df = final_df.sort_values(by=['Company', 'Date'])

    # Chọn thứ tự cột
    cols = ['Date', 'Company', 'Closing price', 'Volume', 'Log return', 'Market return']
    final_df = final_df[cols]

    # Lưu file
    output_path = "/content/drive/MyDrive/chay nhap/price and log - Sheet1.csv"
    final_df.to_csv(output_path, index=False)

    print(f"\n🎉 HOÀN TẤT! File mới đã lưu tại: {output_path}")
    print(f"Kích thước: {final_df.shape}")
    print(f"Số lượng công ty: {final_df['Company'].nunique()}")
    print(final_df.head())
else:
    print("❌ Thất bại hoàn toàn. Kiểm tra lại kết nối mạng.")

# Collect macroeconomic

In [ ]:
# ================= GOOGLE COLAB SETUP =================
import requests
from bs4 import BeautifulSoup
import csv
import time
import re
from datetime import datetime
from urllib.parse import urljoin, quote
from google.colab import drive
import os
import json


# --------- CẤU HÌNH TỪ KHÓA (NHIỀU KEYWORD) ----------
KEYWORDS = [
    "lạm phát",
    "gdp",
    "lãi suất",
    "tỉ giá",
    "tỷ giá",
    "tăng trưởng kinh tế",
    "tín dụng"
]


FROM_DATE_STR = "01/01/2023"
TO_DATE_STR   = "30/06/2024"
FROM_DATE = datetime.strptime(FROM_DATE_STR, "%d/%m/%Y")
TO_DATE   = datetime.strptime(TO_DATE_STR, "%d/%m/%Y")


# ====== SETUP GOOGLE DRIVE ======
drive.mount('/content/drive')


OUT_DIR = "/content/drive/MyDrive/cafef_scrape"
os.makedirs(OUT_DIR, exist_ok=True)


# Output file chung
OUTPUT_FILE = os.path.join(OUT_DIR, f"cafef_MULTIKEY_06-2019_06-2024.csv")
CHECKPOINT_META = os.path.join(OUT_DIR, f"cafef_MULTIKEY_checkpoint.json")


# Cấu hình request
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}


TITLE_SELECTOR = "#admWrapsite > div.main > div.detail-section.adm-mainsection > div.wap-content.wp1100 > div > div.left_cate.totalcontentdetail > h1"
DATE_SELECTOR  = "#admWrapsite > div.main > div.detail-section.adm-mainsection > div.wap-content.wp1100 > div > div.left_cate.totalcontentdetail > div.sharemxh.topshare > p > span"


PAGES_PER_CHECKPOINT = 5
MAX_PAGES = None




# ======================================================
#                 CHECKPOINT FUNCTIONS
# ======================================================
def save_checkpoint(meta):
    with open(CHECKPOINT_META, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


def load_checkpoint():
    if os.path.exists(CHECKPOINT_META):
        with open(CHECKPOINT_META, "r", encoding="utf-8") as f:
            return json.load(f)
    return None




# ======================================================
#                CSV APPEND FUNCTION
# ======================================================
def append_rows_to_csv(path, rows):
    header = ["Keyword", "Tiêu đề", "Ngày đăng", "URL"]
    file_exists = os.path.exists(path)
    with open(path, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(header)
        writer.writerows(rows)




# ======================================================
#         SCRAPE TITLE + DATE FROM AN ARTICLE
# ======================================================
def scrape_article_get_date_and_title(url):
    try:
        res = requests.get(url, headers=HEADERS, timeout=20)
    except:
        return None

    if res.status_code != 200:
        return None


    soup = BeautifulSoup(res.content, "html.parser")
    title_el = soup.select_one(TITLE_SELECTOR)
    date_el  = soup.select_one(DATE_SELECTOR)


    if not title_el or not date_el:
        return None


    title = title_el.get_text(strip=True)
    date_text = date_el.get_text(" ", strip=True)


    m = re.search(r"\d{2}[-/]\d{2}[-/]\d{4}", date_text)
    if not m:
        return None

    date_raw = m.group()
    fmt = "%d-%m-%Y" if "-" in date_raw else "%d/%m/%Y"


    try:
        date_obj = datetime.strptime(date_raw, fmt)
    except:
        return None


    return title, date_obj




# ======================================================
#                    MAIN CRAWLER
# ======================================================
meta = load_checkpoint()
start_page = 1
seen_urls = set()
total_saved = 0


if meta:
    print("🔁 Resume từ checkpoint.")
    start_page = meta.get("last_page", 1) + 1
    seen_urls   = set(meta.get("seen_urls", []))
else:
    print("🚀 Bắt đầu crawl multi-keywords.")




page = start_page
pages_since_checkpoint = 0


while True:
    if MAX_PAGES and page > MAX_PAGES:
        break


    print(f"\n================= PAGE {page} =================")


    # ==================================================================
    #     1 PAGE — chạy nhiều keyword → ghép tất cả URL lại
    # ==================================================================
    urls_in_page = []
    for kw in KEYWORDS:
        url_kw = f"https://cafef.vn/tim-kiem/trang-{page}.chn?keywords={quote(kw)}"
        print(f"[SEARCH] {kw} → {url_kw}")


        try:
            r = requests.get(url_kw, headers=HEADERS, timeout=20)
        except:
            continue
        if r.status_code != 200:
            continue


        soup = BeautifulSoup(r.content, "html.parser")
        h3s = soup.find_all("h3")
        for h3 in h3s:
            a = h3.find("a")
            if a and a.get("href"):
                full = urljoin("https://cafef.vn", a["href"])
                if full not in seen_urls:
                    urls_in_page.append((kw, full))
                    seen_urls.add(full)


    if not urls_in_page:
        print("➡️ Không còn URL mới. Dừng.")
        break


    print(f"➡️ Tổng URL mới tại trang {page}: {len(urls_in_page)}")


    # ==================================================================
    #     SCRAPE CHI TIẾT BÀI
    # ==================================================================
    page_rows = []
    count_out_of_range = 0


    for kw, u in urls_in_page:
        print("   → Vào:", u)
        res = scrape_article_get_date_and_title(u)
        if not res:
            continue


        title, date_obj = res


        if date_obj < FROM_DATE:
            count_out_of_range += 1
            continue


        if date_obj > TO_DATE:
            continue


        page_rows.append([kw, title, date_obj.strftime("%d/%m/%Y"), u])
        time.sleep(0.5)


    # write page
    if page_rows:
        append_rows_to_csv(OUTPUT_FILE, page_rows)
        total_saved += len(page_rows)
        print(f"✔️ Ghi {len(page_rows)} bài.")
    else:
        print("Không có bài hợp lệ.")


    # checkpoint
    pages_since_checkpoint += 1
    if pages_since_checkpoint >= PAGES_PER_CHECKPOINT:
        save_checkpoint({
            "last_page": page,
            "seen_urls": list(seen_urls)
        })
        print("🔖 Saved checkpoint.")
        pages_since_checkpoint = 0


    if count_out_of_range == len(urls_in_page):
        print("➡️ Tất cả bài đều cũ hơn FROM_DATE → dừng.")
        break


    page += 1
    time.sleep(1)


print("\n===== DONE =====")
print("Output:", OUTPUT_FILE)
print("Tổng bài hợp lệ:", total_saved)





# Merge all datas

In [ ]:
import pandas as pd
import glob
import os

# ===============================
# 1. Cấu hình đường dẫn
# ===============================
# Lưu ý: Hãy đảm bảo đường dẫn trỏ đúng tới file của bạn trong Drive
folder_path = "/content/drive/MyDrive/chay nhap"

# Đường dẫn cụ thể (Sửa lại nếu cần thiết)
firm_file = os.path.join(folder_path, "firm sentiment - Sheet1 (1).csv")
macro_file = os.path.join(folder_path, "macro_sentiment - Sheet1 (1).csv")
price_file = os.path.join(folder_path, "price and log - Sheet1 (1).csv")

# ===============================
# 2. Hàm hỗ trợ (Utils)
# ===============================
def fix_date(x):
    if pd.isna(x):
        return pd.NaT
    x = str(x).strip()
    for fmt in ("%Y-%m-%d", "%d-%m-%Y", "%Y/%m/%d", "%d/%m/%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            continue
    try:
        return pd.to_datetime(x, dayfirst=True)
    except:
        return pd.NaT

def find_col(df, keywords):
    """Tìm tên cột trong df chứa từ khóa"""
    for col in df.columns:
        for kw in keywords:
            if kw.lower() in col.lower():
                return col
    return None

# ===============================
# 3. Xử lý từng file
# ===============================

# --- 3.1 PRICE FILE (Đã đối chiếu với file bạn gửi) ---
print("--- Loading Price File ---")
price_df = pd.read_csv(price_file)

# Chuẩn hóa tên cột ngay lập tức
price_df = price_df.rename(columns={
    'Log return': 'Log_Return',     # Sửa lỗi space -> underscore
    'Market return': 'Market_Return',
    'Company': 'Ticker'             # Đổi Company -> Ticker để đồng bộ
})

# Xử lý ngày
date_col_price = find_col(price_df, ['date', 'ngày'])
if date_col_price:
    price_df['Date'] = price_df[date_col_price].apply(fix_date)
else:
    print("Cảnh báo: Không tìm thấy cột Date trong file Price")

# --- 3.2 FIRM SENTIMENT FILE ---
print("--- Loading Firm Sentiment File ---")
try:
    firm_df = pd.read_csv(firm_file)

    # Xử lý ngày
    date_col_firm = find_col(firm_df, ['date', 'ngày'])
    firm_df['Date'] = firm_df[date_col_firm].apply(fix_date)

    # Tìm cột Sentiment (Có thể là Sentiment_Score, Score, Sentiment...)
    sent_col_firm = find_col(firm_df, ['sentiment_score', 'score', 'sentiment'])
    print(f"Firm Sentiment Column detected: {sent_col_firm}")

    # Xử lý dữ liệu số
    firm_df[sent_col_firm] = firm_df[sent_col_firm].astype(str).str.replace(',', '.').astype(float)

    # Group by Date + Ticker (Tìm cột Ticker)
    ticker_col = find_col(firm_df, ['ticker', 'symbol', 'stock', 'company'])

    # Tính trung bình
    firm_grouped = firm_df.groupby(['Date', ticker_col])[sent_col_firm].mean().reset_index()

    # Đổi tên chuẩn để merge
    firm_grouped = firm_grouped.rename(columns={
        ticker_col: 'Ticker',
        sent_col_firm: 'Sentiment_Firm'
    })
except Exception as e:
    print(f"Lỗi xử lý file Firm: {e}")
    # Tạo dummy df để code không chết nếu thiếu file
    firm_grouped = pd.DataFrame(columns=['Date', 'Ticker', 'Sentiment_Firm'])

# --- 3.3 MACRO SENTIMENT FILE ---
print("--- Loading Macro Sentiment File ---")
try:
    macro_df = pd.read_csv(macro_file)

    # Xử lý ngày
    date_col_macro = find_col(macro_df, ['date', 'ngày'])
    macro_df['Date'] = macro_df[date_col_macro].apply(fix_date)

    # Tìm cột Sentiment
    sent_col_macro = find_col(macro_df, ['sentiment_score', 'score', 'sentiment'])
    print(f"Macro Sentiment Column detected: {sent_col_macro}")

    # Xử lý dữ liệu số
    macro_df[sent_col_macro] = macro_df[sent_col_macro].astype(str).str.replace(',', '.').astype(float)

    # Group by Date
    macro_grouped = macro_df.groupby('Date')[sent_col_macro].mean().reset_index()
    macro_grouped = macro_grouped.rename(columns={sent_col_macro: 'Sentiment_Macro'})
except Exception as e:
    print(f"Lỗi xử lý file Macro: {e}")
    macro_grouped = pd.DataFrame(columns=['Date', 'Sentiment_Macro'])

# ===============================
# 4. Merge & Finalize
# ===============================

# Merge: Price (gốc) <- Firm (Left) <- Macro (Left)
merged = price_df.merge(firm_grouped, on=['Date', 'Ticker'], how='left')
merged = merged.merge(macro_grouped, on='Date', how='left')

# Đổi tên lại cho khớp với code chạy mô hình (Fixed Effects)
merged = merged.rename(columns={
    'Ticker': 'Company',
    'Sentiment_Firm': 'Company_sentiment_score',
    'Sentiment_Macro': 'Macro_Sentiment'
})

# Lọc thời gian & Data Cleaning
merged = merged.dropna(subset=['Date'])
merged = merged.sort_values(by=['Company', 'Date'])

# Fill NA
merged['Company_sentiment_score'] = merged['Company_sentiment_score'].fillna(0)
merged['Macro_Sentiment'] = merged['Macro_Sentiment'].ffill().fillna(0) # Fill ngày trước đó nếu thiếu

# ===============================
# 5. Lưu kết quả
# ===============================
output_file = os.path.join(folder_path, "Panel_Dataset_Final.csv")
merged.to_csv(output_file, index=False)

print("\n🎉 XONG! File đã lưu tại:", output_file)
print("Các cột trong file cuối cùng:", merged.columns.tolist())
print(merged.head())

# Fixed EfFect Model

In [ ]:
# ============================================
# 1. Cài đặt thư viện & Import
# ============================================
!pip install linearmodels

import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
from google.colab import drive

# ============================================
# 2. Mount Google Drive & Load Data
# ============================================
drive.mount('/content/drive')

# Đường dẫn file (Đảm bảo đúng đường dẫn tới file Panel_Dataset_Final.csv)
file_path = '/content/drive/MyDrive/chay nhap/Panel_Dataset_Final.csv'
df = pd.read_csv(file_path)

print(f"Loaded dataset with shape: {df.shape}")
print("Các cột trong file:", df.columns.tolist())
print(df.head())

# ============================================
# 3. Xử lý dữ liệu cơ bản (Basic Cleaning)
# ============================================

# 3.1. Chuyển dấu phẩy "," thành dấu chấm "." và ép kiểu số
# Cập nhật tên cột theo đúng file: Company_sentiment_score, Macro_Sentiment
cols_to_fix = ['Log_Return', 'Company_sentiment_score',
               'Market_Return', 'Macro_Sentiment', 'Volume']

for col in cols_to_fix:
    if col in df.columns:
        # Chuyển về string, thay thế phẩy, rồi chuyển về số
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3.2. Convert Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# 3.3. Xử lý Missing Values ban đầu
# Company_sentiment_score: NaN -> 0 (Neutral/No news)
if 'Company_sentiment_score' in df.columns:
    df['Company_sentiment_score'] = df['Company_sentiment_score'].fillna(0)

# Macro_Sentiment: NaN -> Median
if 'Macro_Sentiment' in df.columns:
    df['Macro_Sentiment'] = df['Macro_Sentiment'].fillna(df['Macro_Sentiment'].median())

# ============================================
# 4. Xử lý nâng cao & Tạo biến mới (Feature Engineering)
# ============================================

# 4.1. Xóa dữ liệu trùng lặp
# Dùng cột 'Company' thay vì 'Ticker'
print(f"Số dòng trước khi xóa trùng: {len(df)}")
df = df.drop_duplicates(subset=['Company', 'Date'], keep='first')
print(f"Số dòng sau khi xóa trùng: {len(df)}")

# 4.2. Sắp xếp dữ liệu
df = df.sort_values(by=['Company', 'Date'])

# 4.3. Tạo biến Log Volume
df['log_volume'] = np.log(df['Volume'] + 1)

# 4.4. Tạo biến trễ (Lag Return)
# Group theo 'Company'
df['log_return_lag1'] = df.groupby('Company')['Log_Return'].shift(1)

# 4.5. Loại bỏ các dòng NaN sinh ra do quá trình tạo Lag
df_clean = df.dropna(subset=['Log_Return', 'Market_Return', 'log_volume', 'log_return_lag1'])

print("Data cleaning & feature engineering complete.")

# ============================================
# 5. Thống kê & Phân tích tương quan
# ============================================

# 5.1. Thống kê mô tả
desc_cols = ['Log_Return', 'Market_Return', 'Company_sentiment_score', 'log_volume']
print("\n=== 5.1 Descriptive Statistics ===")
# Chỉ lấy những cột tồn tại để tránh lỗi
existing_desc_cols = [c for c in desc_cols if c in df_clean.columns]
print(df_clean[existing_desc_cols].describe().T[['count', 'mean', 'std', 'min', 'max']])

# 5.2. Ma trận tương quan
print("\n=== 5.2 Correlation Matrix ===")
corr_cols = ['Log_Return', 'Company_sentiment_score', 'Macro_Sentiment', 'Market_Return']
existing_corr_cols = [c for c in corr_cols if c in df_clean.columns]
corr_matrix = df_clean[existing_corr_cols].corr()
print(corr_matrix)

# ============================================
# 6. Chạy mô hình Fixed Effects (Panel Regression)
# ============================================

# 6.1. Thiết lập Panel Data (MultiIndex)
# Index là Company và Date
data_panel = df_clean.set_index(['Company', 'Date'])

# 6.2. Định nghĩa biến
y = data_panel['Log_Return']

# Biến độc lập (X)
X = data_panel[['Company_sentiment_score', 'Market_Return',
                'Macro_Sentiment', 'log_volume', 'log_return_lag1']]

# Thêm hằng số (Intercept)
X = sm.add_constant(X)

# 6.3. Chạy mô hình
print("\n=== 6.3 Panel Fixed Effect Results ===")
model = PanelOLS(y, X, entity_effects=True)
fe_results = model.fit(cov_type='clustered', cluster_entity=True)

print(fe_results.summary)

# Full code run Models

In [ ]:
pip install linearmodels statsmodels pandas numpy

In [ ]:
# ======================================================
# === FULL PANEL MODEL NOTEBOOK - 1 CELL TEMPLATE ====
# ======================================================

# ---------- INSTALL ---------
!pip install fpdf linearmodels

# ---------- IMPORT ----------
import sys, os, glob
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.panel import PooledOLS, PanelOLS, RandomEffects, compare
from linearmodels.iv import IVGMM
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
from fpdf import FPDF
import warnings
warnings.filterwarnings('ignore')

# ---------- CONFIG ----------
REPORT_DIR = "REPORT_OUTPUT"
os.makedirs(REPORT_DIR, exist_ok=True)
LOG_FILE = os.path.join(REPORT_DIR, "all_results.txt")

# ---------- REDIRECT PRINT OUTPUT ----------
sys.stdout = open(LOG_FILE, "w")

# ================= AUTO SAVE ALL PLOTS =================
_save_counter = 1
_original_show = plt.show

def auto_save_show():
    global _save_counter
    fname = os.path.join(REPORT_DIR, f"figure_{_save_counter}.png")
    plt.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[Saved plot -> {fname}]")
    _save_counter += 1

plt.show = auto_save_show

# ==================================================
# ============= ==== YOUR MODEL ===================
# ==================================================

# --- 1. DATA LOADING & PREPROCESSING ---
file_path = '/content/drive/MyDrive/Phương pháp lượng/DATA FINAL/Sentiment/Panel_Dataset_Final (1).csv'
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index(['Company', 'Date'])

# --- 2. FEATURE ENGINEERING ---
df['log_volume'] = np.log(df['Volume'] + 1)
df['log_return_lag1'] = df.groupby('Company')['Log_Return'].shift(1)

# --- 3. CLEAN DATA ---
data_clean = df.dropna(subset=['Log_Return','Company_sentiment_score','Market_Return','Macro_Sentiment','log_volume','log_return_lag1'])

# --- 4. DEFINE VARIABLES ---
Y = data_clean['Log_Return']
X = data_clean[['Company_sentiment_score','Market_Return','Macro_Sentiment','log_volume','log_return_lag1']]
X = sm.add_constant(X)

# --- 5. DESCRIPTIVE & CORRELATION ---
print(">>> I. DESCRIPTIVE STATISTICS")
print(data_clean[['Log_Return','Company_sentiment_score','Market_Return','Macro_Sentiment','log_volume','log_return_lag1']].describe())

print("\n>>> II. CORRELATION MATRIX")
corr_matrix = data_clean[['Log_Return','Company_sentiment_score','Market_Return','Macro_Sentiment','log_volume','log_return_lag1']].corr()
print(corr_matrix)

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".4f")
plt.title("Correlation Matrix")
plt.show()

# --- 6. STATIC MODELS ---

# Pooled OLS
model_pols = PooledOLS(Y,X)
res_pols = model_pols.fit(cov_type='clustered',cluster_entity=True)

# Fixed Effects (Entity FE, giống bạn bè)
model_fe = PanelOLS(Y,X,entity_effects=True)
res_fe = model_fe.fit(cov_type='clustered',cluster_entity=True)

# Random Effects
model_re = RandomEffects(Y,X)
res_re = model_re.fit(cov_type='clustered',cluster_entity=True)

print("\n>>> IV. EMPIRICAL RESULTS (Comparison)")
print(compare({'Pooled OLS':res_pols,'Random Effects':res_re,'Fixed Effects':res_fe}))

# --- 7. DIAGNOSTICS ---

print("\n>>> HAUSMAN TEST")
b_fe = res_fe.params
b_re = res_re.params
v_fe = res_fe.cov
v_re = res_re.cov

vars_common = list(set(b_fe.index)&set(b_re.index))
vars_common = [v for v in vars_common if v!='const']

b_diff = b_fe[vars_common]-b_re[vars_common]
v_diff = v_fe.loc[vars_common,vars_common]-v_re.loc[vars_common,vars_common]

chisq = b_diff.T @ np.linalg.inv(v_diff) @ b_diff
pval = 1 - stats.chi2.cdf(chisq,len(vars_common))

print(f"Chi-Square: {chisq:.4f}")
print(f"P-value: {pval:.4f}")

# Breusch-Pagan
print("\n>>> BREUSCH-PAGAN TEST")
bp = het_breuschpagan(res_pols.resids,X)
print("LM:",bp[0],"p-value:",bp[1])

# VIF
print("\n>>> VIF")
vif = pd.DataFrame()
vif["Feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values,i) for i in range(len(X.columns))]
print(vif)

# Durbin Watson
print("\n>>> DURBIN-WATSON")
dw = durbin_watson(res_pols.resids)
print(dw)

# --- 8. PLOT ---
print("\n>>> PLOTTING LOG_RETURN VS MARKET & SENTIMENT")

daily = data_clean.reset_index().groupby('Date').agg({
    'Company_sentiment_score':'mean',
    'Market_Return':'mean',
    'log_return_lag1':'mean'
}).reset_index()

plt.figure(figsize=(14,6))
ax1 = plt.gca()

l1, = ax1.plot(daily['Date'],daily['Company_sentiment_score'],label='Firm Sentiment',color='green')
ax2 = ax1.twinx()
l2, = ax2.plot(daily['Date'],daily['Market_Return'],label='Market Return',color='red',linestyle='--')

ax1.legend([l1,l2],[i.get_label() for i in [l1,l2]], loc='upper right')
plt.title("Relationship Between Firm Sentiment & Market Returns (VN30)")
plt.grid(True,alpha=0.3)
plt.show()

# ==================================================
# ================= MERGE TO PDF ===================
# ==================================================

sys.stdout.close()
sys.stdout = sys.__stdout__

pdf = FPDF()
pdf.set_auto_page_break(True, margin=15)

pdf.add_page()
pdf.set_font("Arial", size=9)

with open(LOG_FILE,"r") as f:
    for line in f:
        pdf.multi_cell(0,5,line)

imgs = glob.glob(f"{REPORT_DIR}/*.png")
for img in imgs:
    pdf.add_page()
    pdf.image(img, x=10, y=25, w=180)

PDF_FILE = "FULL_PANEL_REPORT.pdf"
pdf.output(PDF_FILE)

from google.colab import files
files.download(PDF_FILE)


# Random Effect Model

In [ ]:
# ============================================
# 1. Cài đặt thư viện & Import
# ============================================
!pip install linearmodels

import pandas as pd
import numpy as np
# --- THAY ĐỔI Ở ĐÂY: Import thêm RandomEffects ---
from linearmodels.panel import PanelOLS, RandomEffects
import statsmodels.api as sm
from google.colab import drive

# ============================================
# 2. Mount Google Drive & Load Data
# ============================================
drive.mount('/content/drive')

# Đường dẫn file
file_path = '/content/drive/MyDrive/chay nhap/Panel_Dataset_Final.csv'
df = pd.read_csv(file_path)

print(f"Loaded dataset with shape: {df.shape}")

# ============================================
# 3. Xử lý dữ liệu cơ bản (Basic Cleaning)
# ============================================
cols_to_fix = ['Log_Return', 'Company_sentiment_score',
               'Market_Return', 'Macro_Sentiment', 'Volume']

for col in cols_to_fix:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

if 'Company_sentiment_score' in df.columns:
    df['Company_sentiment_score'] = df['Company_sentiment_score'].fillna(0)

if 'Macro_Sentiment' in df.columns:
    df['Macro_Sentiment'] = df['Macro_Sentiment'].fillna(df['Macro_Sentiment'].median())

# ============================================
# 4. Xử lý nâng cao & Tạo biến mới
# ============================================
df = df.drop_duplicates(subset=['Company', 'Date'], keep='first')
df = df.sort_values(by=['Company', 'Date'])
df['log_volume'] = np.log(df['Volume'] + 1)
df['log_return_lag1'] = df.groupby('Company')['Log_Return'].shift(1)

df_clean = df.dropna(subset=['Log_Return', 'Market_Return', 'log_volume', 'log_return_lag1'])
print("Data cleaning complete.")

# ============================================
# 5. Thống kê & Phân tích tương quan
# ============================================
print("\n=== 5.1 Descriptive Statistics ===")
desc_cols = ['Log_Return', 'Market_Return', 'Company_sentiment_score', 'log_volume']
existing_desc_cols = [c for c in desc_cols if c in df_clean.columns]
print(df_clean[existing_desc_cols].describe().T[['count', 'mean', 'std', 'min', 'max']])

print("\n=== 5.2 Correlation Matrix ===")
corr_cols = ['Log_Return', 'Company_sentiment_score', 'Macro_Sentiment', 'Market_Return']
existing_corr_cols = [c for c in corr_cols if c in df_clean.columns]
print(df_clean[existing_corr_cols].corr())

# ============================================
# 6. Chạy mô hình RANDOM EFFECTS (RE)
# ============================================

# 6.1. Thiết lập Panel Data
data_panel = df_clean.set_index(['Company', 'Date'])

# 6.2. Định nghĩa biến
y = data_panel['Log_Return']
X = data_panel[['Company_sentiment_score', 'Market_Return',
                'Macro_Sentiment', 'log_volume', 'log_return_lag1']]
X = sm.add_constant(X)

# 6.3. Chạy mô hình RE
# --- THAY ĐỔI Ở ĐÂY: Dùng RandomEffects thay vì PanelOLS ---
print("\n=== 6.3 Panel Random Effects Results ===")
model_re = RandomEffects(y, X)
# Lưu ý: RandomEffects không cần entity_effects=True vì nó xử lý sai số ngẫu nhiên

# Fit mô hình
re_results = model_re.fit(cov_type='clustered', cluster_entity=True)

print(re_results.summary)